# Flight Fare Prediction

**Tabular Regression Project** — Predict airline ticket prices.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Flight fare data (local, ~10K+ rows)
Target: `Price` (flight fare in INR)

## 2 · Project Overview

We predict **airline ticket prices** based on features like airline, source, destination, route, departure/arrival time, duration, and number of stops. This is a practical regression problem — fare prediction helps travellers plan budgets and airlines optimize pricing.

The dataset is from Indian domestic flights and contains rich categorical and temporal features that require thoughtful feature engineering.

## 3 · Learning Objectives

1. Parse and engineer features from date/time columns.
2. Handle high-cardinality categorical variables (Route, Airline).
3. Convert duration strings to numeric minutes.
4. Apply gradient-boosting regressors to pricing data.
5. Use LazyPredict and FLAML for model selection.

## 4 · Problem Statement

Predict the **Price** (in INR) of a domestic Indian flight given airline, source, destination, route, departure/arrival times, duration, total stops, and additional info.

## 5 · Why This Project Matters

- **Dynamic pricing** is used by all airlines — understanding price drivers is valuable.
- Flight fare prediction is a classic feature-engineering-heavy regression task.
- Real-world impact: budget planning for travellers, revenue optimization for airlines.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Local file: `Data_Train.xlsx` |
| **Target** | Price (INR) |
| **Key Features** | Airline, Source, Destination, Route, Duration, Total_Stops, Date_of_Journey |
| **Note** | Contains date/time strings requiring parsing |

## 7 · Dataset Source and License Notes

- **Source**: Indian flight booking data, commonly available on Kaggle.
- **License**: Educational/research use.
- **Note**: Prices are in Indian Rupees (INR). Data represents a snapshot in time.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('openpyxl')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Price'
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Loading

In [ ]:
data_file = os.path.join(SAVE_DIR, 'Data_Train.xlsx')
assert os.path.exists(data_file), f'Data file not found: {data_file}'
df = pd.read_excel(data_file)
print(f'Loaded: {df.shape}')
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df[TARGET].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Price Distribution')

if 'Airline' in df.columns:
    df['Airline'].value_counts().head(10).plot.barh(ax=axes[0,1])
    axes[0,1].set_title('Top Airlines')

if 'Source' in df.columns:
    df.groupby('Source')[TARGET].mean().sort_values().plot.barh(ax=axes[1,0])
    axes[1,0].set_title('Avg Price by Source')

if 'Total_Stops' in df.columns:
    df.groupby('Total_Stops')[TARGET].mean().sort_values().plot.barh(ax=axes[1,1])
    axes[1,1].set_title('Avg Price by Stops')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

Parse dates, convert duration to minutes, encode stops and categorical variables.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Parse Date_of_Journey
if 'Date_of_Journey' in df.columns:
    df['Date_of_Journey'] = pd.to_datetime(df['Date_of_Journey'], format='%d/%m/%Y', errors='coerce')
    df['journey_month'] = df['Date_of_Journey'].dt.month
    df['journey_day'] = df['Date_of_Journey'].dt.day
    df = df.drop(columns=['Date_of_Journey'])

# Parse Duration to minutes
if 'Duration' in df.columns:
    def parse_duration(x):
        x = str(x)
        hours = 0; mins = 0
        if 'h' in x:
            hours = int(x.split('h')[0].strip())
            x = x.split('h')[1]
        if 'm' in x:
            mins = int(x.split('m')[0].strip())
        return hours * 60 + mins
    df['duration_min'] = df['Duration'].apply(parse_duration)
    df = df.drop(columns=['Duration'])

# Parse departure/arrival times
for col in ['Dep_Time', 'Arrival_Time']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df[f'{col}_hour'] = df[col].dt.hour
        df = df.drop(columns=[col])

# Total_Stops to numeric
if 'Total_Stops' in df.columns:
    stops_map = {'non-stop': 0, '1 stop': 1, '2 stops': 2, '3 stops': 3, '4 stops': 4}
    df['Total_Stops'] = df['Total_Stops'].map(stops_map)
    df['Total_Stops'] = df['Total_Stops'].fillna(0).astype(int)

# Drop Route and Additional_Info (high cardinality / low info)
for col in ['Route', 'Additional_Info']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Encode remaining categoricals
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 50:
        df = df.drop(columns=[col])
    else:
        df[col] = LabelEncoder().fit_transform(df[col].fillna('Unknown'))

# Fill numeric NaN
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.0f}, MAE={baseline_mae:.0f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Number of stops** and **duration** are typically the strongest price predictors.
- Airlines and routes create significant price variation.
- Journey month/day captures seasonal demand patterns.
- These models could help travellers find optimal booking windows.

## 26 · Limitations

- Single snapshot of prices — not dynamic/real-time.
- Indian domestic flights only.
- No seat class distinction (economy vs business).
- Prices change constantly based on demand, competition, and timing.

## 27 · How to Improve

1. Add days-before-departure feature.
2. Include seat class and booking platform.
3. Time-series of prices for the same route.
4. External data: holidays, events, fuel prices.
5. Try target log-transform for right-skewed prices.

## 28 · Production Considerations

- Real-time price scraping needed.
- Model retraining frequency: weekly or more.
- Integration with booking platforms.
- A/B testing before customer-facing deployment.

## 29 · Common Mistakes

1. Not parsing duration strings properly.
2. Leaving Route as raw text (too many unique values).
3. Not handling the date format correctly (DD/MM/YYYY).
4. Ignoring stops as the dominant feature.

## 30 · Mini Challenge

1. Log-transform the target and compare results.
2. One-hot encode airlines instead of label encoding.
3. Create a 'is_weekend' departure feature.
4. Build route-specific models for top 5 routes.

## 31 · Final Summary

- Flight fare prediction requires substantial feature engineering from raw text/date fields.
- Gradient-boosting models capture non-linear relationships between stops, duration, and price.
- Domain knowledge (airline industry) significantly improves feature engineering.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')